# L06 · DQN and Function Approximation

## Goal

- explain replay and target networks
- stop target gradients
- inspect TD-loss shapes

## Setup

This cell fixes CPU, seed, offline status, and the split hash first. Toy code uses deterministic CPU operations; package trainers retain their strict global default.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L06:toy:42").hexdigest()
print(f"lesson=L06 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L06 language=en profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.10.12 rl_study=0.1.0.dev0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:afdf5bdf3ec7ea09c983beb03612c9488e8db611ddbdd80e0b7b079f922f3d29 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. Position and core equation

⏱ 5 min · 1/3 section · [CORE]

Position: MC/TD/Q-learning → **DQN** → policy gradients

$$L(\theta)=\mathbb{E}\left[\left(Q_\theta(s,a)-\operatorname{stopgrad}(r+\gamma\max_{a'}Q_{\bar\theta}(s',a'))\right)^2\right]$$

DQN approximates a Q table with a network. Replay reduces correlation and reuses experience; a target network is synchronized slowly so the learning target does not move with every gradient step.

### 2. Run with small numbers

⏱ 6 min · 2/3 section · [CORE]

**Predict first:** Should target-network parameters have gradients after `loss.backward()`? Write an answer for 20 seconds, then run the cell.

<details><summary>Show answer</summary>No. The target provides numbers for the learning signal and is not owned by the optimizer.</details>

In [2]:
from rl_study.algorithms.dqn import DQNBatch, DQNNetwork, dqn_loss, hard_update
policy_net = DQNNetwork(16, 4)
target_net = DQNNetwork(16, 4)
hard_update(target_net, policy_net)
dqn_batch = DQNBatch(
    states=torch.tensor([0, 1]), actions=torch.tensor([1, 2]),
    rewards=torch.tensor([-0.01, 1.0]), next_states=torch.tensor([1, 15]),
    terminated=torch.tensor([False, True]), truncated=torch.tensor([False, False])
)
dqn_output = dqn_loss(policy_net, target_net, dqn_batch, gamma=0.99)
dqn_output.loss.backward()
target_has_gradient = any(p.grad is not None for p in target_net.parameters())
print({"loss": round(float(dqn_output.loss.detach()), 4),
       "targets": dqn_output.targets.tolist(),
       "target_has_gradient": target_has_gradient})

{'loss': 0.3481, 'targets': [0.11680736392736435, 1.0], 'target_has_gradient': False}


### 3. Implementation anatomy

⏱ 6 min · 3/3 section · [DEEP DIVE]

**Why this implementation:** A tiny batch first verifies the target-detach contract. Double DQN separates action selection from evaluation to reduce overestimation, but keeps the same gradient ownership.

**Common trap:** Detaching the target tensor is insufficient if target parameters remain in an optimizer and can move under another loss. Check freezing and optimizer membership too. Regression tests: `test_dqn_target_detached`.

**Checkpoint:** Continue when you can explain just one printed value.

## Checks

In [3]:
assert not target_has_gradient and torch.isfinite(dqn_output.loss)
print("checks=passed")

checks=passed


**Recall:** How do the instabilities addressed by replay and target networks differ? Answer in one or two sentences.

## Mistakes I Revisit

- Assuming a finite loss proves the implementation is correct.
- Merging `terminated` with `truncated`, or prompt with action.
- Turning one tiny seed into an algorithm ranking.

## 60-Second Recap

- **Run conclusion:** The loss and targets were finite, with `target_has_gradient=False`. This output validates the frozen-target boundary rather than training quality.
- Executable checks: `test_dqn_target_detached`.
- The output is a fixed-seed toy run, not a paper-scale result.

## Next Steps

1. L07 moves policy probabilities directly toward reward instead of fitting action values.
2. Break one `[CORE]` assertion and read the failure.
3. Open the package test and connect the notebook equation to its production guard.

## Sources

- `dqn-2013` — `docs/sources.yml`